In [ ]:
# Extract_State_City_County.py
# 6/17/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import sys
import random
import requests
import math
import urllib3
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import numpy as np

def right(s, amount):
    return s[-amount:]

def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])

def get_city_county_json(data):
        # Step 2: Split into lines and clean up
    decoded = file_content.decode("utf-8", errors="ignore")
    lines = decoded.strip().splitlines()
    fields = ["State", "State Code", "Place Code", "Place Name", "Place Type", "Functional Status", "County"]
    # Step 4: Convert to list of dictionaries
    records = []
    for line in lines:
        parts = line.strip().split("|")
        if len(parts) == len(fields):
            record = dict(zip(fields, parts))
            records.append(record)
    capture_json = json.dumps(records, indent=2)
    return capture_json

def extract_state_city_county_data(capture_json_raw):
    global field_list_set,field_list
    capture_json=json.loads(capture_json_raw)
    collection_df=pd.DataFrame(capture_json)
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    state_city_county_df=collection_df[field_list]
    return state_city_county_df

def random_sleep(call_count): 
    global sleep_time_pause,sleep_range,pause_count
    if call_count%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'
}


DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"
fileInputEvents='States_Abv_Codes'
fileOutputEvents='States_Cities_Counties'

fileInput=DirMain+"DataFiles\\"+fileInputEvents+".csv"
fileOut=DirMain+"Data\\"+fileOutputEvents+".csv"
fileOutInt=DirMain+"Data\\"+fileOutputEvents+"_Interval.csv"
state_codes_df=pd.read_csv(fileInput)

field_list=['State','Place Name','County']
field_list_set=set(field_list)

url_state_code="https://www2.census.gov/geo/docs/reference/codes/files/st"
url_suffix="_places.txt"

sleep_range=[3,8]
sleep_time_pause=1*60
pause_count=100
get_counter=0
state_city_county_df=pd.DataFrame([])
for index, row in state_codes_df.iterrows():
    state_abrv=row['Abbreviation'].lower()
    state_code=row['code']
    state_code_str=right('0'+str(state_code),2)
    state_code_text=right('0'+str(state_code),2)
    state_name=row['State']
    url=url_state_code+state_code_str+'_'+state_abrv+url_suffix
    response = requests.get(url)   
    file_content = response.content 
    capture_json=get_city_county_json(file_content)
    city_county_df=extract_state_city_county_data(capture_json)
    city_county_df['state_name']=state_name

    new_field_list=['state_name','State','Place Name', 'County']
    city_county_df = city_county_df[new_field_list]
    state_city_county_df = pd.concat([state_city_county_df,city_county_df]) 
    state_city_county_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    
    print(f"** {state_name}**" )
    get_counter+=1
    sleep_time=random_sleep(get_counter)       
    time.sleep(sleep_time) 
    # print(type(file_content))
    # input('')
state_city_county_df['County'] = state_city_county_df['County'].replace(' County', '', regex=True)

city_split=state_city_county_df['Place Name'].str.rsplit(' ')
state_city_county_df['city']=city_split.str[0:-1]
state_city_county_df['city']=state_city_county_df['city'].str.join(' ')

state_city_county_df=state_city_county_df[['state_name',	'State','city',	'County']]		
state_city_county_df.to_csv(fileOut,mode="w", index=False,header=True)
print('*** Processing Complete ***')



In [142]:
state_city_county_df=state_city_county_df[['state_name',	'State','city',	'County']]	
state_city_county_df.to_csv(fileOut,mode="w", index=False,header=True)
print('*** Processing Complete ***')

*** Processing Complete ***


In [135]:
state_city_county_df=pd.read_csv(fileOutInt)
state_city_county_df['city']=city_split.str[0:-1]
state_city_county_df['city']
state_city_county_df['city']=state_city_county_df['city'].str.join(' ')
state_city_county_df['city']
# state_city_county_df.to_csv(fileOut,mode="w", index=False,header=True)

0                    Abanda
1                 Abbeville
2                Adamsville
3                   Addison
4                     Akron
                ...        
41156    Woods Landing-Jelm
41157               Worland
41158                Wright
41159                 Yoder
41160             Y-O Ranch
Name: city, Length: 41161, dtype: object